In [1]:
# Dir paths
import os
BASE_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
PARENT_DIR = os.path.dirname(BASE_DIR)
MESSAGES_DIR = os.path.join(PARENT_DIR, "data","messages")
MESSAGES_DIR

'd:\\python\\AI\\pynb\\data\\messages'

In [2]:
## Read CSVs
message_csv_path = os.path.join(MESSAGES_DIR, "sample_msg_testing.csv")
groups_csv_path = os.path.join(MESSAGES_DIR, "groups.csv")
group_members_csv_path = os.path.join(MESSAGES_DIR, "group_members.csv")
images_csv_path = os.path.join(MESSAGES_DIR, "images.csv")
voice_notes_csv_path = os.path.join(MESSAGES_DIR, "voice_notes.csv")

In [3]:
## pandas data frames
import pandas as pd

message_df = pd.read_csv(message_csv_path)
groups_df = pd.read_csv(groups_csv_path)
group_members_df = pd.read_csv(group_members_csv_path)
images_df = pd.read_csv(images_csv_path)
voice_notes_df = pd.read_csv(voice_notes_csv_path)
group_conversation_df = message_df[message_df["conversation_type"] == "group"]

group_conversation_df.drop_duplicates().shape


(17, 11)

In [ ]:
def enhance_file_path(path:str) -> str:
    if not pd.isna(path):
        path = path.replace("/", "\\")
        return os.path.join(MESSAGES_DIR, path)
    return path


def get_group_message_data(group_conversation_df:pd.DataFrame) -> df:
    df = (
        group_conversation_df
        .merge(groups_df, on="group_id", how="left")
        .merge(group_members_df, left_on=["group_id", "sender_user_id"], right_on=["group_id", "user_id"], how="left")
        .merge(images_df, left_on="media_id", right_on="image_id", how="left")
        .merge(voice_notes_df, left_on="media_id", right_on="voice_note_id", how="left")
    )

    df["file_path"] = df["file_path_x"].combine_first(df["file_path_y"])

    df["file_path"] = df["file_path"].apply(enhance_file_path)

    #df["media_for_ai"] = df["file_path"].apply(cook_img_part)


    return df.to_dict(orient="records")



In [ ]:
message_data_list = get_group_message_data(group_conversation_df)

In [6]:
from google import genai
GL_GEN_AI_API_KEY=os.environ.get('GL_GEN_AI_API_KEY')

client = genai.Client(api_key=GL_GEN_AI_API_KEY)

d:\python\AI\pynb\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [9]:

for message_data in message_data_list[11:16]:
    prompt = f"""
    You are a message triage classifier.

Input:
{message_data}

Task:
Classify the message and return **only one valid Python dictionary**.

Schema:
{{
    "message_id": "<copy from input>",
    "user_id": "<copy from input>",
    "conversation_type": "<copy from input>",
    "group_id": "<copy from input>",
    "business_id": "<copy from input>",
    "sender_user_id": "<copy from input>",
    "created_at": "<copy from input>",
    "media_type": "<copy from input>",
    "media_id": "<copy from input>",
    "forwarded_count": "<copy from input>",
    "action": "<notify|digest|mute>",
    "message_type": "<personal|urgent|event|payment|business_update|promotion|greeting|forward|spam|scam|unknown>",
    "reason": "<max 20 words>",
    "confidence": <float between 0.0 and 1.0>,
    "evidence_message_ids": ["<message_id>", ...]
}}

Classification Rules

Action
- notify: Legitimate and requires immediate attention.
- digest: Legitimate but not urgent.
- mute: Promotion, spam, repetitive, suspicious, phishing, scam, or other low-value content.

Message Types
- personal: Personal conversation.
- urgent: Legitimate time-sensitive message.
- event: Meeting, booking, reminder, invitation.
- payment: Banking, bills, invoices, refunds, transactions.
- business_update: Official work or business communication.
- promotion: Advertisements, offers, discounts, marketing.
- greeting: Wishes or congratulations.
- forward: Forwarded or chain messages.
- spam: Unwanted unsolicited or repetitive content.
- scam: Phishing, fraud, fake security alerts, credential theft, impersonation, suspicious links, fake urgency.
- unknown: None of the above.

Priority (highest to lowest)

1. Scam
   - Any phishing, fraud, suspicious URL, fake OTP/security alert, credential request, impersonation, or fake urgency.
   - Always classify as:
       action = "mute"
       message_type = "scam"

2. Promotion
   - Marketing, offers, advertisements.
   - action = "mute"

3. Spam
   - Unwanted, repetitive, or irrelevant messages.
   - action = "mute"

4. Otherwise
   - Choose the best matching message_type.
   - Select notify only if the message is both legitimate and immediately actionable; otherwise use digest.

Reason
- Maximum 20 words.
- State the primary reason for the classification.

Confidence
Return a float between 0.00 and 1.00.

Guide:
- 0.95 - 1.00: Very high confidence
- 0.80 - 0.94: Strong confidence
- 0.60 - 0.79: Moderate confidence
- <0.60: Uncertain

Evidence
- Populate `evidence_message_ids` only with message IDs from the messages that influenced the decision.

Output Requirements
- Copy all metadata fields exactly as provided.
- Do not invent or modify values.
- Choose exactly one `action`.
- Choose exactly one `message_type`.
- Return only the json.
- Do not include Markdown, explanations, or extra text.
"""
    print(f"message_id : {message_data["message_id"]} | message_type : {message_data["media_type"]}")
    if message_data["media_type"] in ("voice", "image"):
        uploaded_file = client.files.upload(file=message_data['file_path'])
        input_data = [
            {"type": "text", "text": prompt},
            {
                "type": "audio" if message_data["media_type"] == "voice" else message_data["media_type"],
                "uri": uploaded_file.uri,
                "mime_type": uploaded_file.mime_type
            }
        ]
    else:
        input_data = [
            {"type": "text", "text": prompt},
        ]


    interaction = client.interactions.create(
        model="gemini-2.5-flash",
        input=input_data
    )
    print(interaction.output_text)

message_id : msg_020 | message_type : nan
{"message_id": "msg_020", "user_id": "u_005", "conversation_type": "group", "group_id": "group_005", "business_id": null, "sender_user_id": "u_050", "created_at": "7/31/2026 8:04", "media_type": null, "media_id": null, "forwarded_count": 0, "action": "mute", "message_type": "scam", "reason": "Fake support alert attempting to phish password and OTP with false urgency, threatening profile blockage.", "confidence": 0.99, "evidence_message_ids": ["msg_020"]}
message_id : msg_041 | message_type : voice
{"message_id": "msg_041", "user_id": "u_024", "conversation_type": "group", "group_id": "group_008", "business_id": "nan", "sender_user_id": "u_041", "created_at": "7/31/2026 11:09", "media_type": "voice", "media_id": "vn_001", "forwarded_count": 0, "action": "digest", "message_type": "personal", "reason": "The voice note explicitly states 'nothing urgent' and describes personal activities, indicating a casual update.", "confidence": 0.98, "evidence_m